# Prostate Zonal Segmentation — nnU-Net v2 Baseline

**Input**: 3-channel biparametric MRI (T2W + ADC + HBV)  
**Output**: Multi-class zonal segmentation (Background / PZ / TZ)  
**Training**: RUMC + ZGT (5-fold CV)  
**External Test**: PCNN (never used for training)  

## Checkpointing Strategy
Every time-consuming step is cached to Google Drive:  
- ✅ Data Conversion → `PI-CAI_nnUNet_Raw_Zonal.zip`  
- ✅ Preprocessing → `PI-CAI_nnUNet_Preprocessed_Zonal.zip`  
- ✅ Splits → `splits_final_zonal.json`  
- ✅ Training → auto-resume from Drive checkpoints  

---
## Cell 1: Mount Drive & Setup Environment

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone the Zonal Segmentation repo
!git clone https://github.com/HemishJain09/Zonal_Segmentation.git /content/ZonalSeg 2>/dev/null || \
  (cd /content/ZonalSeg && git pull)

# Clone nnU-Net from source (for full control)
!git clone https://github.com/MIC-DKFZ/nnUNet.git /content/nnUNet 2>/dev/null
!cd /content/nnUNet && pip install -e . -q

# Install additional dependencies
!pip install -q nibabel pandas scikit-learn tqdm scipy

print('\n✅ Environment setup complete!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nnunetv2 (pyproject.toml) ... done

✅ Environment setup complete!


---
## Cell 2: Set Environment Variables

In [3]:
import os

# nnU-Net environment variables
os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/drive/MyDrive/ZonalSeg_Results/nnUNet_results'

# Create directories
!mkdir -p $nnUNet_raw $nnUNet_preprocessed $nnUNet_results

# Configuration
PICAI_DIR = '/content/drive/MyDrive/PI-CAI_pre-processed'  # Adjust path
MARKSHEET = '/content/drive/MyDrive/marksheet.csv'  # Adjust path
DRIVE_CACHE_DIR = '/content/drive/MyDrive/ZonalSeg_Cache'
RESULTS_DIR = '/content/drive/MyDrive/ZonalSeg_Results'
DATASET_ID = '501'
DATASET_NAME = 'Dataset501_ZonalSeg'

!mkdir -p {DRIVE_CACHE_DIR} {RESULTS_DIR}

print(f'PICAI_DIR:    {PICAI_DIR}')
print(f'DRIVE_CACHE:  {DRIVE_CACHE_DIR}')
print(f'RESULTS_DIR:  {RESULTS_DIR}')
print(f'\n✅ Environment variables set!')

PICAI_DIR:    /content/drive/MyDrive/PI-CAI_pre-processed
DRIVE_CACHE:  /content/drive/MyDrive/ZonalSeg_Cache
RESULTS_DIR:  /content/drive/MyDrive/ZonalSeg_Results

✅ Environment variables set!


---
# PHASE 1: Dataset Characterization
Run this FIRST to verify zonal masks exist and check label encoding.

In [4]:
# Phase 1: Characterize the dataset (quick check on 20 cases)
!python /content/ZonalSeg/data/characterize_dataset.py \
    --picai_dir {PICAI_DIR} \
    --marksheet {MARKSHEET} \
    --output_dir {RESULTS_DIR} \
    --max_cases 20

print('\n📄 Check the dataset report above!')
print('Pay attention to:')
print('  1. Are zonal masks found?')
print('  2. What are the unique label values?')
print('  3. Are all modalities present for all cases?')

PI-CAI Dataset Characterization Report

## 1. Folder Structure Discovery

  t2w             → t2/ (1500 files)
  zonal           → zonal_masks/ (1500 files)

## 2. Centre Distribution

Traceback (most recent call last):
  File "/content/ZonalSeg/data/characterize_dataset.py", line 318, in <module>
    characterize_dataset(args.picai_dir, args.marksheet, args.output_dir, args.max_cases)
  File "/content/ZonalSeg/data/characterize_dataset.py", line 138, in characterize_dataset
    marksheet = pd.read_csv(marksheet_path)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/

---
# PHASE 2: Sanity Check (10 cases, 5 epochs)
Verify the entire pipeline works before committing GPU hours.

### Phase 2 — Step 1: Convert 10 cases

In [ ]:
# Sanity Check: Convert 10 cases
!python /content/ZonalSeg/data/convert_to_nnunet.py \
    --picai_dir {PICAI_DIR} \
    --nnunet_raw $nnUNet_raw \
    --marksheet {MARKSHEET} \
    --train_centers RUMC ZGT \
    --max_cases 10

### Phase 2 — Step 2: Verify conversion

In [ ]:
# Verify conversion output
import json

dataset_json_path = f'{os.environ["nnUNet_raw"]}/{DATASET_NAME}/dataset.json'
with open(dataset_json_path) as f:
    dj = json.load(f)

print('Dataset JSON:')
print(json.dumps(dj, indent=2))

n_images = len(os.listdir(f'{os.environ["nnUNet_raw"]}/{DATASET_NAME}/imagesTr'))
n_labels = len(os.listdir(f'{os.environ["nnUNet_raw"]}/{DATASET_NAME}/labelsTr'))

print(f'\nImages: {n_images} files (expected: {dj["numTraining"]} × 3 = {dj["numTraining"] * 3})')
print(f'Labels: {n_labels} files (expected: {dj["numTraining"]})')

assert n_images == dj['numTraining'] * 3, f'❌ Image count mismatch!'
assert n_labels == dj['numTraining'], f'❌ Label count mismatch!'
print('\n✅ Data conversion verified!')

### Phase 2 — Step 3: Plan & Preprocess

In [ ]:
# Plan and preprocess (sanity check)
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity

### Phase 2 — Step 4: Generate Splits

In [ ]:
# Generate patient-level splits (sanity check)
!python /content/ZonalSeg/data/generate_splits.py \
    --nnunet_raw $nnUNet_raw \
    --nnunet_preprocessed $nnUNet_preprocessed \
    --marksheet {MARKSHEET} \
    --train_centers RUMC ZGT

### Phase 2 — Step 5: Train 5 Epochs (Sanity Check)

In [ ]:
# Modify nnU-Net source to run only 5 epochs
!sed -i 's/self.num_epochs = 1000/self.num_epochs = 5/' \
    /content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py

# Verify the change
!grep 'self.num_epochs' /content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py | head -1

print('\n✅ Epochs set to 5 for sanity check')

In [ ]:
# Train 5 epochs (sanity check)
!nnUNetv2_train {DATASET_ID} 3d_fullres 0

In [ ]:
# Verify sanity check passed
import glob

checkpoint_dir = f'{os.environ["nnUNet_results"]}/{DATASET_NAME}'
checkpoints = glob.glob(f'{checkpoint_dir}/**/*.pth', recursive=True)

if checkpoints:
    print(f'✅ Sanity check PASSED! Found {len(checkpoints)} checkpoint(s):')
    for cp in checkpoints:
        print(f'  {cp}')
else:
    print('❌ No checkpoints found. Check the training output above.')

# Revert epochs to production value
!sed -i 's/self.num_epochs = 5/self.num_epochs = 250/' \
    /content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py
!grep 'self.num_epochs' /content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py | head -1
print('\n✅ Epochs reverted to 250 for production training')

---
# PHASE 3: Full Production Training
Only proceed after Phase 2 sanity check passes!

**Checkpointing**: Every step caches to Drive. If Colab disconnects, re-run and it will skip completed steps.

### Phase 3 — Step 1: Smart Data Conversion (with cache)

In [ ]:
import os
import subprocess

RAW_CACHE = f'{DRIVE_CACHE_DIR}/PI-CAI_nnUNet_Raw_Zonal.zip'
PREPROCESSED_CACHE = f'{DRIVE_CACHE_DIR}/PI-CAI_nnUNet_Preprocessed_Zonal.zip'
SPLITS_CACHE = f'{DRIVE_CACHE_DIR}/splits_final_zonal.json'

nnunet_raw_local = os.environ['nnUNet_raw']
nnunet_preprocessed_local = os.environ['nnUNet_preprocessed']

# ============================================================
# CHECKPOINT 1: Raw Conversion
# ============================================================
if os.path.exists(RAW_CACHE):
    print('📦 CHECKPOINT HIT: Raw conversion cache found on Drive!')
    print('   Unzipping to local SSD (skip 20+ min conversion)...')
    subprocess.run([
        'unzip', '-q', '-o', RAW_CACHE, '-d', '/content/'
    ], check=True)
    print('   ✅ Raw data restored from cache!')
else:
    print('❌ CHECKPOINT MISS: No raw conversion cache. Running full conversion...')
    
    # Clean any partial data
    subprocess.run(['rm', '-rf', f'{nnunet_raw_local}/{DATASET_NAME}'], check=False)
    
    # Convert ALL RUMC+ZGT cases
    subprocess.run([
        'python', '/content/ZonalSeg/data/convert_to_nnunet.py',
        '--picai_dir', PICAI_DIR,
        '--nnunet_raw', nnunet_raw_local,
        '--marksheet', MARKSHEET,
        '--train_centers', 'RUMC', 'ZGT'
    ], check=True)
    
    # Cache to Drive
    print('\n💾 Saving raw conversion to Drive cache...')
    subprocess.run([
        'zip', '-r', '-q', RAW_CACHE, f'{nnunet_raw_local}/{DATASET_NAME}'
    ], check=True)
    print(f'   ✅ Cached to: {RAW_CACHE}')

print('\n✅ Step 1 complete: Raw data ready!')

### Phase 3 — Step 2: Smart Preprocessing (with cache)

In [ ]:
# ============================================================
# CHECKPOINT 2: Preprocessing
# ============================================================
if os.path.exists(PREPROCESSED_CACHE):
    print('📦 CHECKPOINT HIT: Preprocessed cache found on Drive!')
    print('   Unzipping to local SSD (skip 1-2 hour preprocessing)...')
    subprocess.run([
        'unzip', '-q', '-o', PREPROCESSED_CACHE, '-d', '/content/'
    ], check=True)
    print('   ✅ Preprocessed data restored from cache!')
else:
    print('❌ CHECKPOINT MISS: No preprocessed cache. Running preprocessing...')
    !nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity
    
    # Cache to Drive
    print('\n💾 Saving preprocessed data to Drive cache...')
    subprocess.run([
        'zip', '-r', '-q', PREPROCESSED_CACHE, 
        f'{nnunet_preprocessed_local}/{DATASET_NAME}'
    ], check=True)
    print(f'   ✅ Cached to: {PREPROCESSED_CACHE}')

print('\n✅ Step 2 complete: Preprocessed data ready!')

### Phase 3 — Step 3: Smart Splits Generation (with cache)

In [ ]:
# ============================================================
# CHECKPOINT 3: Splits
# ============================================================
import shutil

local_splits = f'{nnunet_preprocessed_local}/{DATASET_NAME}/splits_final.json'

if os.path.exists(SPLITS_CACHE) and not os.path.exists(local_splits):
    print('📦 CHECKPOINT HIT: Splits cache found on Drive!')
    os.makedirs(os.path.dirname(local_splits), exist_ok=True)
    shutil.copy2(SPLITS_CACHE, local_splits)
    print(f'   ✅ Splits restored from cache!')
elif os.path.exists(local_splits):
    print('✅ Splits already exist locally (from preprocessing cache).')
else:
    print('❌ CHECKPOINT MISS: Generating fresh splits...')
    !python /content/ZonalSeg/data/generate_splits.py \
        --nnunet_raw $nnUNet_raw \
        --nnunet_preprocessed $nnUNet_preprocessed \
        --marksheet {MARKSHEET} \
        --train_centers RUMC ZGT
    
    # Cache to Drive
    shutil.copy2(local_splits, SPLITS_CACHE)
    print(f'   ✅ Splits cached to: {SPLITS_CACHE}')

# Verify
with open(local_splits) as f:
    splits = json.load(f)
print(f'\nSplits: {len(splits)} folds')
for i, fold in enumerate(splits):
    print(f'  Fold {i}: train={len(fold["train"])}, val={len(fold["val"])}')

print('\n✅ Step 3 complete: Splits ready!')

### Phase 3 — Step 4: Set Training Epochs

In [ ]:
# Set training to 250 epochs (balanced speed vs. convergence)
!sed -i 's/self.num_epochs = [0-9]*/self.num_epochs = 250/' \
    /content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py

!grep 'self.num_epochs' /content/nnUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py | head -1
print('\n✅ Training set to 250 epochs')

### Phase 3 — Step 5: Train (Auto-Resume from Checkpoint)

In [ ]:
# ============================================================
# CHECKPOINT 4: Training (nnU-Net auto-resumes from last checkpoint)
# ============================================================
# nnU-Net automatically detects existing checkpoints in nnUNet_results
# and resumes training from the last saved epoch.
# Because nnUNet_results points to Google Drive, this works across
# Colab disconnections!

!nnUNetv2_train {DATASET_ID} 3d_fullres 0

---
# PHASE 4: Inference & Evaluation
After training completes, run predictions and compute metrics.

In [ ]:
# Run predictions on validation set
# (nnU-Net predicts on the fold's validation cases automatically)
!nnUNetv2_predict \
    -i $nnUNet_raw/{DATASET_NAME}/imagesTr \
    -o /content/predictions \
    -d {DATASET_ID} \
    -c 3d_fullres \
    -f 0

In [ ]:
# Compute evaluation metrics
!python /content/ZonalSeg/evaluation/compute_metrics.py \
    --predictions /content/predictions \
    --ground_truth $nnUNet_raw/{DATASET_NAME}/labelsTr \
    --output_dir {RESULTS_DIR} \
    --mode internal

In [ ]:
# Error analysis
!python /content/ZonalSeg/evaluation/error_analysis.py \
    --metrics_csv {RESULTS_DIR}/metrics_internal.csv \
    --predictions /content/predictions \
    --ground_truth $nnUNet_raw/{DATASET_NAME}/labelsTr \
    --images $nnUNet_raw/{DATASET_NAME}/imagesTr \
    --output_dir {RESULTS_DIR}/error_analysis

---
## Summary
| Step | Status | Cache |
|------|--------|-------|
| Dataset Characterization | Phase 1 | Report on Drive |
| Data Conversion | Phase 3 Step 1 | `PI-CAI_nnUNet_Raw_Zonal.zip` |
| Preprocessing | Phase 3 Step 2 | `PI-CAI_nnUNet_Preprocessed_Zonal.zip` |
| Splits | Phase 3 Step 3 | `splits_final_zonal.json` |
| Training | Phase 3 Step 5 | Auto-resume via Drive |
| Evaluation | Phase 4 | Results on Drive |